# ЛР № 3. Профилирование данных, проверка качества и подготовка слоя `staging` (Movie Industry)

Исполняемый ноутбук участка `raw → staging` для набора данных *Movie Industry* (`movies.csv`, каталог полнометражных кинофильмов с метаданными IMDb).

- **Основной вопрос из [Project Brief](../docs/project_brief.md):** как в период четырёх полных десятилетий с 1980 по 2019 год (`[1980; 2020)`) различаются количество выпущенных фильмов, медианный коэффициент кассовой отдачи (`gross / budget`), доля кассово окупившихся картин (`gross >= budget`) и медианная оценка IMDb (`score`) в разрезе основных жанров (`genre`), возрастных категорий (`rating`) и десятилетий выпуска?
- **Гранулярность (смысл одной строки):** один полнометражный кинофильм.
- **Бизнес-ключ дедупликации:** отдельного числового идентификатора в исходном файле нет, а названия у ремейков и одноимённых картин разных лет повторяются, поэтому уникальность релиза определяется составным ключом `(lower(trim(name)), year)`.

Перед запуском убедитесь, что стенд SeaweedFS запущен (`docker compose up -d`), выберите ядро из `.venv` проекта и выполните все ячейки сверху вниз.

<a id="input"></a>
## Шаг 1. Входной снимок из слоя `raw` и подключение к SeaweedFS

Префикс `ingested_on=2026-10-24` фиксирует логическую дату загрузки файла через DAG `ingest_raw` из ЛР № 2, тогда как аналитический период исследования ограничен четырьмя полными десятилетиями `1980–2019` гг. (`[1980; 2020)`).

Ячейка ниже определяет корень репозитория, загружает параметры подключения к S3 из `.env` без вывода секретов, переиспользует функции `configure_s3` и `reader_for` из `src/analysis/read_raw.py` и проверяет прямое чтение CSV-объекта средствами DuckDB.

In [1]:
import os
import sys
from html import escape
from pathlib import Path

import boto3
import duckdb
from dotenv import load_dotenv
from IPython.display import HTML, display

PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "pyproject.toml").is_file()
    and (candidate / "src/analysis/read_raw.py").is_file()
)
load_dotenv(PROJECT_ROOT / ".env", override=False)

required_env = ("S3_ENDPOINT", "S3_ACCESS_KEY", "S3_SECRET_KEY")
missing_env = [var for var in required_env if not os.environ.get(var)]
if missing_env:
    msg = "Отсутствуют переменные окружения S3: " + ", ".join(missing_env)
    raise ValueError(msg)

src_dir = str(PROJECT_ROOT / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
from analysis.read_raw import configure_s3, reader_for  # noqa: E402

RAW_S3_URI = "s3://raw/movie_industry/ingested_on=2026-10-24/movies.csv"
STUDY_YEAR_MIN = 1980
STUDY_YEAR_MAX = 2019
STAGING_S3_PREFIX = "s3://staging/movie_industry/source_ingested_on=2026-10-24"


def sql_literal(text: str) -> str:
    return "'" + text.replace("'", "''") + "'"


def render_table(headers: list[str], rows: list[tuple[object, ...]]) -> None:
    head_html = "".join(f"<th>{escape(str(col))}</th>" for col in headers)
    body_html = "".join(
        "<tr>" + "".join(f"<td>{escape(str(val))}</td>" for val in row) + "</tr>"
        for row in rows
    )
    display(HTML(f"<table><thead><tr>{head_html}</tr></thead>"
                 f"<tbody>{body_html}</tbody></table>"))


s3_endpoint = os.environ["S3_ENDPOINT"]
s3_endpoint_url = (
    s3_endpoint if s3_endpoint.startswith("http") else f"http://{s3_endpoint}"
)
s3_client = boto3.client(
    "s3",
    endpoint_url=s3_endpoint_url,
    aws_access_key_id=os.environ["S3_ACCESS_KEY"],
    aws_secret_access_key=os.environ["S3_SECRET_KEY"],
    region_name="us-east-1",
)
raw_key = RAW_S3_URI.removeprefix("s3://raw/")
raw_head_before = s3_client.head_object(Bucket="raw", Key=raw_key)
raw_meta_before = (raw_head_before["ContentLength"], raw_head_before["ETag"])

raw_reader = reader_for(RAW_S3_URI)
conn = duckdb.connect()
configure_s3(conn)
conn.execute(
    f"CREATE OR REPLACE TEMP VIEW raw_movies AS "
    f"SELECT * FROM {raw_reader}({sql_literal(RAW_S3_URI)}, "
    f"header = true, hive_partitioning = false)"
)

render_table(
    ["Входной S3-путь", "Целевой период", "Выходной префикс", "Байт в raw"],
    [(
        RAW_S3_URI,
        f"{STUDY_YEAR_MIN}–{STUDY_YEAR_MAX}",
        STAGING_S3_PREFIX,
        raw_meta_before[0],
    )],
)
preview_rows = conn.execute("""
    SELECT name, year, genre, rating, score, votes, budget, gross, runtime
    FROM raw_movies
    LIMIT 5
""").fetchall()
render_table(
    ["name", "year", "genre", "rating", "score", "votes",
     "budget", "gross", "runtime"],
    preview_rows,
)

Входной S3-путь,Целевой период,Выходной префикс,Байт в raw
s3://raw/movie_industry/ingested_on=2026-10-24/movies.csv,1980–2019,s3://staging/movie_industry/source_ingested_on=2026-10-24,1344929


name,year,genre,rating,score,votes,budget,gross,runtime
The Shining,1980,Drama,R,8.4,927000.0,19000000.0,46998772.0,146.0
The Blue Lagoon,1980,Adventure,R,5.8,65000.0,4500000.0,58853106.0,104.0
Star Wars: Episode V - The Empire Strikes Back,1980,Action,PG,8.7,1200000.0,18000000.0,538375067.0,124.0
Airplane!,1980,Comedy,PG,7.7,221000.0,3500000.0,83453539.0,88.0
Caddyshack,1980,Comedy,R,7.3,108000.0,6000000.0,39846344.0,98.0


<a id="profile"></a>
## Шаг 2. Профиль исходных данных

Параметр `hive_partitioning = false` при чтении CSV отключает автоматическое добавление виртуальной колонки `ingested_on` из S3-пути, поэтому схема в точности отражает 15 собственных полей файла `movies.csv` (без него DuckDB возвращает 16 полей).

Для профилирования до любых преобразований отобраны поля, непосредственно участвующие в метриках, срезах и ключе дедупликации из `Project Brief`:
- составной бизнес-ключ и временная ось: `name`, `year`;
- ключевые аналитические поля: `genre`, `rating`, `score`, `votes`, `budget`, `gross`, `runtime`.

In [2]:
raw_row_count = conn.execute("SELECT count(*) FROM raw_movies").fetchone()[0]
raw_schema = {
    col_name: col_type for col_name, col_type, *_ in conn.execute(
        "DESCRIBE raw_movies"
    ).fetchall()
}
render_table(["Строк в raw", "Полей в CSV"], [(raw_row_count, len(raw_schema))])

profile_fields = (
    "name",
    "year",
    "genre",
    "rating",
    "score",
    "votes",
    "budget",
    "gross",
    "runtime",
)
field_profile_rows = []
for col in profile_fields:
    missing_rows = conn.execute(
        f"SELECT count(*) FROM raw_movies "
        f"WHERE {col} IS NULL OR trim(CAST({col} AS VARCHAR)) = ''"
    ).fetchone()[0]
    sample_vals = conn.execute(
        f"SELECT DISTINCT {col} FROM raw_movies "
        f"WHERE {col} IS NOT NULL AND trim(CAST({col} AS VARCHAR)) <> '' "
        f"ORDER BY 1 LIMIT 2"
    ).fetchall()
    missing_share = f"{missing_rows / raw_row_count:.3%}" if raw_row_count else "N/A"
    field_profile_rows.append((
        col,
        raw_schema[col],
        ", ".join(str(item[0]) for item in sample_vals),
        raw_row_count,
        missing_rows,
        missing_share,
    ))
render_table(
    ["Поле", "Тип DuckDB", "Примеры значений", "Проверено",
     "Пропусков", "Доля пропусков"],
    field_profile_rows,
)

Строк в raw,Полей в CSV
7668,15


Поле,Тип DuckDB,Примеры значений,Проверено,Пропусков,Доля пропусков
name,VARCHAR,"'71, 'Round Midnight",7668,0,0.000%
year,BIGINT,"1980, 1981",7668,0,0.000%
genre,VARCHAR,"Action, Adventure",7668,0,0.000%
rating,VARCHAR,"Approved, G",7668,77,1.004%
score,DOUBLE,"1.9, 2.1",7668,3,0.039%
votes,DOUBLE,"7.0, 14.0",7668,3,0.039%
budget,DOUBLE,"3000.0, 5000.0",7668,2171,28.312%
gross,DOUBLE,"309.0, 596.0",7668,189,2.465%
runtime,DOUBLE,"55.0, 63.0",7668,4,0.052%


In [3]:
min_year, max_year = conn.execute(
    "SELECT min(year), max(year) FROM raw_movies"
).fetchone()
composite_key_repeats = conn.execute("""
    SELECT count(*) - count(DISTINCT struct_pack(n := lower(trim(name)), y := year))
    FROM raw_movies
    WHERE name IS NOT NULL AND trim(name) <> '' AND year IS NOT NULL
""").fetchone()[0]
title_only_repeats = conn.execute("""
    SELECT count(*) - count(DISTINCT lower(trim(name)))
    FROM raw_movies
    WHERE name IS NOT NULL AND trim(name) <> ''
""").fetchone()[0]
exact_row_repeats = conn.execute("""
    SELECT count(*) - (SELECT count(*) FROM (SELECT DISTINCT * FROM raw_movies))
    FROM raw_movies
""").fetchone()[0]
render_table(
    ["Мин. год", "Макс. год", "Повторов ключа (name, year)",
     "Повторов только name", "Полных дубликатов строк"],
    [(min_year, max_year, composite_key_repeats,
      title_only_repeats, exact_row_repeats)],
)

decade_distribution = conn.execute(f"""
    SELECT
        CASE
            WHEN year BETWEEN {STUDY_YEAR_MIN} AND {STUDY_YEAR_MAX}
                THEN concat((year // 10) * 10, '–', (year // 10) * 10 + 9)
            ELSE concat(CAST(year AS VARCHAR), ' (вне полных десятилетий)')
        END AS period_bucket,
        count(*) AS films
    FROM raw_movies
    GROUP BY 1
    ORDER BY min(year)
""").fetchall()
render_table(["Период (по полю year)", "Фильмов в raw"], decade_distribution)

genre_counts = conn.execute("""
    SELECT genre, count(*) AS films
    FROM raw_movies
    GROUP BY 1
    ORDER BY films DESC, genre
""").fetchall()
top5_genres = genre_counts[:5]
other_genres_sum = sum(row[1] for row in genre_counts[5:])
render_table(
    ["Основной жанр (genre)", "Фильмов в raw"],
    [*top5_genres, (f"Остальные жанры ({len(genre_counts) - 5})", other_genres_sum)],
)

Мин. год,Макс. год,"Повторов ключа (name, year)",Повторов только name,Полных дубликатов строк
1980,2020,0,157,0


Период (по полю year),Фильмов в raw
1980–1989,1643
1990–1999,2000
2000–2009,2000
2010–2019,2000
2020 (вне полных десятилетий),25


Основной жанр (genre),Фильмов в raw
Comedy,2245
Action,1705
Drama,1518
Crime,551
Biography,443
Остальные жанры (14),1206


### Значимые наблюдения по профилю

1. **Неполный 2020 год на правой границе периода:** при наполнении каждого из полных десятилетий `1980–1989` (`1 643` фильма) и `1990–2019` (ровно по `2 000` фильмов на десятилетие) на `2020` год приходится лишь `25` записей начала следующего десятилетия `[2020; 2030)`. Включение этих 25 строк создало бы несопоставимую группу при сравнении десятилетий, поэтому целевой интервал `1980–2019` проверяется как критическое правило отбора.
2. **Структурная неполнота финансов (`28.312 %` без `budget`) и неоднородность `rating`:** отсутствие бюджета или кассовых сборов означает нераскрытую в источнике коммерческую отчётность (преимущественно у независимых и зарубежных релизов), а не повреждение карточки фильма. Исключение таких строк в карантин исказило бы общее количество картин и медианный рейтинг `score` по жанрам, поэтому для финансовой пары и нестандартных возрастных сертификатов уместно действие `flag`, а `quarantine` — только для критических дефектов периода, ключа, оценки и хронометража.

*Способ подсчёта дубликатов:* число повторов вычисляется как разность между общим числом проверенных строк и количеством уникальных комбинаций `(lower(trim(name)), year)` (либо `SELECT DISTINCT *` для полных повторов). Одно название `name` без года повторяется `157` раз из-за ремейков (*Hercules*, *The Lion King* и др.), но составной ключ `(lower(trim(name)), year)` уникален во всём наборе (`0` повторов).

<a id="rules"></a>
## Шаг 3. Правила качества данных

Ниже реализованы шесть правил для профилированных полей, совместно закрывающие все четыре обязательных измерения качества: уникальность по составному бизнес-ключу, полноту значимых атрибутов, допустимость значений (диапазоны и справочник MPAA) и межполевую согласованность (`score` ↔ `votes`, пара `budget` ↔ `gross`).

Единица порога во всех правилах — допустимое число нарушений в наборе (`count <= 0`), заданное отдельной переменной в коде каждого правила. Сравнение с порогом оценивает весь снимок, а действие (`quarantine` или `flag`) применяется к каждой нарушившей записи.

In [4]:
rule_results: dict[str, tuple[object, ...]] = {}


def record_rule(
    rule_id: str,
    checked: int,
    violations: int,
    max_violations: int,
    severity: str,
    action: str,
) -> None:
    share = f"{violations / checked:.3%}" if checked else "N/A"
    status = (
        ("Превышен" if violations > max_violations else "Не превышен")
        if checked
        else "N/A"
    )
    summary_row = (
        rule_id,
        checked,
        violations,
        share,
        f"count <= {max_violations}",
        status,
        severity,
        action,
    )
    rule_results[rule_id] = summary_row
    render_table(
        ["Правило", "Проверено", "Нарушений", "Доля",
         "Порог", "Сравнение", "Критичность", "Действие"],
        [summary_row],
    )

### Правило 1. `unique_movie_key`

- **Поля:** составной бизнес-ключ `(name, year)`.
- **Проверяемые строки:** все записи набора (`raw_movies`).
- **Условие:** `name` не пусто после удаления пробелов, `year` заполнено, а пара `(lower(trim(name)), year)` встречается в снимке ровно один раз.
- **Порог:** число нарушений `count <= 0` (переменная `max_invalid_movie_keys`).
- **Критичность:** `critical`.
- **Действие:** `quarantine`.
- **Связь с задачей:** единица анализа в `Project Brief` — отдельный кинофильм; пустой или задвоенный ключ исказит подсчёт количества релизов (`count(*)`) и расчёт медиан по жанрам и десятилетиям.

In [5]:
max_invalid_movie_keys = 0
r1_checked, r1_violations = conn.execute("""
    WITH keyed AS (
        SELECT
            name,
            year,
            count(*) OVER (PARTITION BY lower(trim(name)), year) AS key_cnt
        FROM raw_movies
    )
    SELECT
        count(*),
        count(*) FILTER (
            WHERE name IS NULL OR trim(name) = '' OR year IS NULL OR key_cnt > 1
        )
    FROM keyed
""").fetchone()
record_rule(
    "unique_movie_key",
    r1_checked,
    r1_violations,
    max_invalid_movie_keys,
    "critical",
    "quarantine",
)

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Критичность,Действие
unique_movie_key,7668,0,0.000%,count <= 0,Не превышен,critical,quarantine


### Правило 2. `year_in_study_period`

- **Поле:** `year`.
- **Проверяемые строки:** записи с заполненным годом выпуска (`year IS NOT NULL`).
- **Условие:** `STUDY_YEAR_MIN <= year <= STUDY_YEAR_MAX` (`1980 <= year <= 2019`).
- **Порог:** число нарушений `count <= 0` (переменная `max_outside_study_period`).
- **Критичность:** `critical`.
- **Действие:** `quarantine`.
- **Связь с задачей:** сравнение десятилетий в `Project Brief` определено на четырёх полных десятилетиях `[1980; 2020)`; релизы 2020 года относятся к началу следующего десятилетия `[2020; 2030)` и не должны смешиваться с полными декадами.

In [6]:
max_outside_study_period = 0
r2_checked, r2_violations = conn.execute(f"""
    SELECT
        count(*),
        count(*) FILTER (WHERE year < {STUDY_YEAR_MIN} OR year > {STUDY_YEAR_MAX})
    FROM raw_movies
    WHERE year IS NOT NULL
""").fetchone()
record_rule(
    "year_in_study_period",
    r2_checked,
    r2_violations,
    max_outside_study_period,
    "critical",
    "quarantine",
)

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Критичность,Действие
year_in_study_period,7668,25,0.326%,count <= 0,Превышен,critical,quarantine


### Правило 3. `valid_score_and_votes`

- **Поля (межполевая согласованность и диапазон):** `score`, `votes`.
- **Проверяемые строки:** все записи набора (`raw_movies`).
- **Условие:** оба поля одновременно заполнены, оценка лежит в шкале IMDb `0 <= score <= 10`, а число голосов положительно и целочисленно (`votes >= 1` и `votes = floor(votes)`), поскольку зрительский рейтинг не может существовать при нуле голосов.
- **Порог:** число нарушений `count <= 0` (переменная `max_invalid_score_votes`).
- **Критичность:** `critical`.
- **Действие:** `quarantine`.
- **Связь с задачей:** защищает расчёт `median(score)` и `median(votes)` из основного и второго уточняющего вопросов `Project Brief`.

In [7]:
max_invalid_score_votes = 0
r3_checked, r3_violations = conn.execute("""
    SELECT
        count(*),
        count(*) FILTER (
            WHERE score IS NULL
               OR score < 0
               OR score > 10
               OR votes IS NULL
               OR votes < 1
               OR votes <> floor(votes)
        )
    FROM raw_movies
""").fetchone()
record_rule(
    "valid_score_and_votes",
    r3_checked,
    r3_violations,
    max_invalid_score_votes,
    "critical",
    "quarantine",
)

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Критичность,Действие
valid_score_and_votes,7668,3,0.039%,count <= 0,Превышен,critical,quarantine


### Правило 4. `positive_runtime`

- **Поле:** `runtime`.
- **Проверяемые строки:** все записи набора (`raw_movies`).
- **Условие:** длительность фильма заполнена, строго положительна и выражена целым числом минут (`runtime IS NOT NULL AND runtime > 0 AND runtime = floor(runtime)`).
- **Порог:** число нарушений `count <= 0` (переменная `max_invalid_runtime`).
- **Критичность:** `critical`.
- **Действие:** `quarantine`.
- **Связь с задачей:** уточняющий вопрос 2 в `Project Brief` сопоставляет `median(runtime)` по десятилетиям и странам; восстанавливать пустой хронометраж по догадке недопустимо.

In [8]:
max_invalid_runtime = 0
r4_checked, r4_violations = conn.execute("""
    SELECT
        count(*),
        count(*) FILTER (
            WHERE runtime IS NULL OR runtime <= 0 OR runtime <> floor(runtime)
        )
    FROM raw_movies
""").fetchone()
record_rule(
    "positive_runtime",
    r4_checked,
    r4_violations,
    max_invalid_runtime,
    "critical",
    "quarantine",
)

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Критичность,Действие
positive_runtime,7668,4,0.052%,count <= 0,Превышен,critical,quarantine


### Правило 5. `complete_positive_financials`

- **Поля (связанная финансовая пара):** `budget`, `gross`.
- **Проверяемые строки:** все записи набора (`raw_movies`).
- **Условие:** одновременно заполнены и строго положительны производственный бюджет и валовые сборы (`budget > 0 AND gross > 0`).
- **Порог:** число нарушений `count <= 0` (переменная `max_incomplete_financials`).
- **Критичность:** `warning`.
- **Действие:** `flag` (булев признак `quality_incomplete_financials` в `staging`).
- **Связь с задачей:** метрики `median(gross / budget)` и доля `gross >= budget` считаются только при наличии обоих положительных полей, но сами фильмы без раскрытого бюджета должны остаться в `staging` для подсчёта числа релизов и медианных оценок `score`.

In [9]:
max_incomplete_financials = 0
r5_checked, r5_violations = conn.execute("""
    SELECT
        count(*),
        count(*) FILTER (
            WHERE budget IS NULL OR budget <= 0 OR gross IS NULL OR gross <= 0
        )
    FROM raw_movies
""").fetchone()
record_rule(
    "complete_positive_financials",
    r5_checked,
    r5_violations,
    max_incomplete_financials,
    "warning",
    "flag",
)

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Критичность,Действие
complete_positive_financials,7668,2232,29.108%,count <= 0,Превышен,warning,flag


### Правило 6. `standard_mpaa_rating`

- **Поле:** `rating`.
- **Проверяемые строки:** все записи набора (`raw_movies`).
- **Условие:** возрастной сертификат заполнен и принадлежит основной кинопрокатной шкале MPAA (`G`, `PG`, `PG-13`, `R`, `NC-17`).
- **Порог:** число нарушений `count <= 0` (переменная `max_nonstandard_mpaa`).
- **Критичность:** `warning`.
- **Действие:** `flag` (булев признак `quality_nonstandard_rating` в `staging`).
- **Связь с задачей:** в срезах по возрастному рейтингу из `Project Brief` необходимо явно отделять стандартные прокатные категории MPAA от пропусков и внерейтинговых/телевизионных меток (`Not Rated`, `Unrated`, `TV-MA`, `X` и др.), не исключая такие картины из жанровых и временных сравнений.

In [10]:
max_nonstandard_mpaa = 0
r6_checked, r6_violations = conn.execute("""
    SELECT
        count(*),
        count(*) FILTER (
            WHERE rating IS NULL
               OR trim(rating) NOT IN ('G', 'PG', 'PG-13', 'R', 'NC-17')
        )
    FROM raw_movies
""").fetchone()
record_rule(
    "standard_mpaa_rating",
    r6_checked,
    r6_violations,
    max_nonstandard_mpaa,
    "warning",
    "flag",
)

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Критичность,Действие
standard_mpaa_rating,7668,431,5.621%,count <= 0,Превышен,warning,flag


In [11]:
render_table(
    ["Правило", "Проверено", "Нарушений", "Доля",
     "Порог", "Сравнение", "Критичность", "Действие"],
    list(rule_results.values()),
)

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Критичность,Действие
unique_movie_key,7668,0,0.000%,count <= 0,Не превышен,critical,quarantine
year_in_study_period,7668,25,0.326%,count <= 0,Превышен,critical,quarantine
valid_score_and_votes,7668,3,0.039%,count <= 0,Превышен,critical,quarantine
positive_runtime,7668,4,0.052%,count <= 0,Превышен,critical,quarantine
complete_positive_financials,7668,2232,29.108%,count <= 0,Превышен,warning,flag
standard_mpaa_rating,7668,431,5.621%,count <= 0,Превышен,warning,flag


<a id="prepare"></a>
## Шаг 4. Подготовка `staging` и `quarantine` и запись в SeaweedFS

**Нормализация в `staging.parquet` (без догадок и без бизнес-агрегаций):**
- строковые поля очищаются от краевых пробелов через `NULLIF(trim(...), '')`;
- целочисленные по смыслу атрибуты приводятся к целым типам (`year` → `INTEGER`, `votes` → `BIGINT`, `runtime` → `INTEGER`), вещественные — к `DOUBLE` (`score`, `budget`, `gross`);
- добавляются булевы признаки предупреждений `quality_incomplete_financials` и `quality_nonstandard_rating`.

**Правила формирования `quarantine.parquet`:**
- карантин имеет приоритет над `flag`; каждая исключённая запись сохраняется ровно один раз со всеми 15 исходными полями без принудительных преобразований;
- колонка `quality_rule_ids` перечисляет сработавшие критические правила через точку с запятой (`;`), а `quarantine_reason` — читаемое пояснение через `; `;
- схема файла строго типизирована даже при нуле исключённых записей.

In [12]:
conn.execute(f"""
    CREATE OR REPLACE TEMP VIEW evaluated_movies AS
    SELECT
        *,
        (name IS NULL OR trim(name) = '' OR year IS NULL
         OR count(*) OVER (PARTITION BY lower(trim(name)), year) > 1) AS v_key,
        COALESCE(year < {STUDY_YEAR_MIN} OR year > {STUDY_YEAR_MAX},
                 false) AS v_period,
        (score IS NULL OR score < 0 OR score > 10
         OR votes IS NULL OR votes < 1
         OR votes <> floor(votes)) AS v_score_votes,
        (runtime IS NULL OR runtime <= 0
         OR runtime <> floor(runtime)) AS v_runtime,
        (budget IS NULL OR budget <= 0
         OR gross IS NULL OR gross <= 0) AS v_financials,
        (rating IS NULL
         OR trim(rating) NOT IN ('G', 'PG', 'PG-13', 'R', 'NC-17')) AS v_mpaa
    FROM raw_movies
""")

conn.execute("""
    CREATE OR REPLACE TEMP VIEW staging_prepared AS
    SELECT
        NULLIF(trim(name), '') AS name,
        NULLIF(trim(rating), '') AS rating,
        NULLIF(trim(genre), '') AS genre,
        CAST(year AS INTEGER) AS year,
        NULLIF(trim(released), '') AS released,
        CAST(score AS DOUBLE) AS score,
        CAST(votes AS BIGINT) AS votes,
        NULLIF(trim(director), '') AS director,
        NULLIF(trim(writer), '') AS writer,
        NULLIF(trim(star), '') AS star,
        NULLIF(trim(country), '') AS country,
        CAST(budget AS DOUBLE) AS budget,
        CAST(gross AS DOUBLE) AS gross,
        NULLIF(trim(company), '') AS company,
        CAST(runtime AS INTEGER) AS runtime,
        v_financials AS quality_incomplete_financials,
        v_mpaa AS quality_nonstandard_rating
    FROM evaluated_movies
    WHERE NOT (v_key OR v_period OR v_score_votes OR v_runtime)
""")

conn.execute("""
    CREATE OR REPLACE TEMP VIEW quarantine_prepared AS
    SELECT
        * EXCLUDE (v_key, v_period, v_score_votes, v_runtime,
                   v_financials, v_mpaa),
        array_to_string(
            list_filter([
                CASE WHEN v_key THEN 'unique_movie_key' END,
                CASE WHEN v_period THEN 'year_in_study_period' END,
                CASE WHEN v_score_votes THEN 'valid_score_and_votes' END,
                CASE WHEN v_runtime THEN 'positive_runtime' END
            ], item -> item IS NOT NULL),
            ';'
        ) AS quality_rule_ids,
        array_to_string(
            list_filter([
                CASE WHEN v_key
                     THEN 'Пустой или дублирующийся ключ (name, year)' END,
                CASE WHEN v_period
                     THEN 'Год выпуска вне полных десятилетий 1980–2019' END,
                CASE WHEN v_score_votes
                     THEN 'Некорректная или неполная пара (score, votes)' END,
                CASE WHEN v_runtime
                     THEN 'Отсутствует или неположителен хронометраж runtime' END
            ], item -> item IS NOT NULL),
            '; '
        ) AS quarantine_reason
    FROM evaluated_movies
    WHERE v_key OR v_period OR v_score_votes OR v_runtime
""")

split_counts = [
    ("raw_movies", conn.execute("SELECT count(*) FROM raw_movies").fetchone()[0]),
    ("staging_prepared",
     conn.execute("SELECT count(*) FROM staging_prepared").fetchone()[0]),
    ("quarantine_prepared",
     conn.execute("SELECT count(*) FROM quarantine_prepared").fetchone()[0]),
]
render_table(["Представление", "Число строк"], split_counts)

quarantine_samples = conn.execute("""
    (SELECT name, year, score, votes, runtime, quality_rule_ids, quarantine_reason
     FROM quarantine_prepared
     WHERE year <= 2019
     ORDER BY year, name LIMIT 2)
    UNION ALL
    (SELECT name, year, score, votes, runtime, quality_rule_ids, quarantine_reason
     FROM quarantine_prepared
     WHERE quality_rule_ids LIKE '%;%'
     ORDER BY name LIMIT 2)
""").fetchall()
render_table(
    ["name", "year", "score", "votes", "runtime",
     "quality_rule_ids", "quarantine_reason"],
    quarantine_samples,
)

flagged_samples = conn.execute("""
    SELECT name, year, rating, budget, gross,
           quality_incomplete_financials, quality_nonstandard_rating
    FROM staging_prepared
    WHERE quality_incomplete_financials AND quality_nonstandard_rating
    ORDER BY year, name
    LIMIT 3
""").fetchall()
render_table(
    ["name", "year", "rating", "budget", "gross",
     "quality_incomplete_financials", "quality_nonstandard_rating"],
    flagged_samples,
)

Представление,Число строк
raw_movies,7668
staging_prepared,7640
quarantine_prepared,28


name,year,score,votes,runtime,quality_rule_ids,quarantine_reason
Saw: The Final Chapter,2010,5.6,93000.0,None,positive_runtime,Отсутствует или неположителен хронометраж runtime
The Wolfman,2010,5.8,104000.0,None,positive_runtime,Отсутствует или неположителен хронометраж runtime
It's Just Us,2020,None,None,120.0,year_in_study_period;valid_score_and_votes,"Год выпуска вне полных десятилетий 1980–2019; Некорректная или неполная пара (score, votes)"
Love by Drowning,2020,None,None,121.0,year_in_study_period;valid_score_and_votes,"Год выпуска вне полных десятилетий 1980–2019; Некорректная или неполная пара (score, votes)"


name,year,rating,budget,gross,quality_incomplete_financials,quality_nonstandard_rating
Babylon,1980,None,None,109749.0,True,True
Mantis Fist Fighter,1980,None,None,None,True,True
The King and the Mockingbird,1980,Not Rated,None,167451.0,True,True


In [13]:
ingested_date = RAW_S3_URI.split("/ingested_on=", 1)[1].split("/", 1)[0]
expected_prefix = f"s3://staging/movie_industry/source_ingested_on={ingested_date}"
if STAGING_S3_PREFIX != expected_prefix:
    msg = f"Ожидался префикс {expected_prefix}, получен {STAGING_S3_PREFIX}"
    raise ValueError(msg)

staging_rel_prefix = STAGING_S3_PREFIX.removeprefix("s3://staging/")
written_objects = []
for target_name, view_name in (
    ("staging", "staging_prepared"),
    ("quarantine", "quarantine_prepared"),
):
    object_key = f"{staging_rel_prefix}/{target_name}.parquet"
    target_uri = f"s3://staging/{object_key}"
    s3_client.delete_object(Bucket="staging", Key=object_key)
    conn.execute(
        f"COPY (SELECT * FROM {view_name} "
        f"ORDER BY year NULLS LAST, lower(trim(name)) NULLS LAST) "
        f"TO {sql_literal(target_uri)} (FORMAT PARQUET)"
    )
    obj_head = s3_client.head_object(Bucket="staging", Key=object_key)
    written_objects.append((target_name, target_uri, obj_head["ContentLength"]))

render_table(["Набор", "S3 URI", "Размер Parquet (байт)"], written_objects)

Набор,S3 URI,Размер Parquet (байт)
staging,s3://staging/movie_industry/source_ingested_on=2026-10-24/staging.parquet,519377
quarantine,s3://staging/movie_industry/source_ingested_on=2026-10-24/quarantine.parquet,6681


<a id="report"></a>
## Шаг 5. Краткая сводка для `docs/data_quality_report.md`

- Сумма срабатываний четырёх критических правил в `raw` равна `0 + 25 + 3 + 4 = 32`, но в `quarantine.parquet` записано **`28` уникальных фильмов**: четыре релиза 2020 года нарушают сразу два критических правила (`3` картины без оценок/голосов и `1` картина — *Saving Mbango* — без `runtime`) и сохраняются по одному разу с перечислением обоих идентификаторов в `quality_rule_ids`.
- Среди `28` карантинных записей у `19` также отсутствовали полные финансы и у `12` — стандартный сертификат MPAA, поэтому в `staging.parquet` (`7 640` строк) остаётся **`2 213`** записей с `quality_incomplete_financials = true` и **`419`** записей с `quality_nonstandard_rating = true`.

<a id="verify"></a>
## Шаг 6. Проверка сохранённых Parquet-файлов из S3

Ниже оба записанных файла читаются заново напрямую из бакета `staging` (`hive_partitioning = false`). Проверяются:
1. баланс строк `raw = staging + quarantine`;
2. отсутствие любых нарушений четырёх критических правил (включая уникальность составного ключа `(name, year)`) в сохранённом `staging.parquet`;
3. точное соответствие обоих булевых флагов фактическим условиям на полях `staging.parquet`;
4. заполненность `quality_rule_ids` и `quarantine_reason` у всех строк карантина;
5. неизменность размера и `ETag` исходного объекта в бакете `raw`.

In [14]:
for target_name in ("staging", "quarantine"):
    parquet_uri = f"{STAGING_S3_PREFIX}/{target_name}.parquet"
    conn.execute(
        f"CREATE OR REPLACE TEMP VIEW saved_{target_name} AS "
        f"SELECT * FROM read_parquet({sql_literal(parquet_uri)}, "
        f"hive_partitioning = false)"
    )

saved_staging_rows = conn.execute("SELECT count(*) FROM saved_staging").fetchone()[0]
saved_quarantine_rows = conn.execute(
    "SELECT count(*) FROM saved_quarantine"
).fetchone()[0]
saved_staging_cols = len(conn.execute("DESCRIBE saved_staging").fetchall())
saved_quarantine_cols = len(conn.execute("DESCRIBE saved_quarantine").fetchall())

critical_in_staging = conn.execute(f"""
    WITH keyed AS (
        SELECT
            *,
            count(*) OVER (PARTITION BY lower(trim(name)), year) AS key_cnt
        FROM saved_staging
    )
    SELECT count(*)
    FROM keyed
    WHERE name IS NULL OR trim(name) = '' OR year IS NULL OR key_cnt > 1
       OR year < {STUDY_YEAR_MIN} OR year > {STUDY_YEAR_MAX}
       OR score IS NULL OR score < 0 OR score > 10
       OR votes IS NULL OR votes < 1
       OR runtime IS NULL OR runtime <= 0
""").fetchone()[0]

mismatched_fin_flag = conn.execute("""
    SELECT count(*) FROM saved_staging
    WHERE quality_incomplete_financials IS DISTINCT FROM
          (budget IS NULL OR budget <= 0 OR gross IS NULL OR gross <= 0)
""").fetchone()[0]
mismatched_mpaa_flag = conn.execute("""
    SELECT count(*) FROM saved_staging
    WHERE quality_nonstandard_rating IS DISTINCT FROM
          (rating IS NULL OR rating NOT IN ('G', 'PG', 'PG-13', 'R', 'NC-17'))
""").fetchone()[0]
saved_fin_flagged = conn.execute(
    "SELECT count(*) FROM saved_staging WHERE quality_incomplete_financials"
).fetchone()[0]
saved_mpaa_flagged = conn.execute(
    "SELECT count(*) FROM saved_staging WHERE quality_nonstandard_rating"
).fetchone()[0]
empty_quarantine_reasons = conn.execute("""
    SELECT count(*) FROM saved_quarantine
    WHERE quality_rule_ids IS NULL OR trim(quality_rule_ids) = ''
       OR quarantine_reason IS NULL OR trim(quarantine_reason) = ''
""").fetchone()[0]

raw_head_after = s3_client.head_object(Bucket="raw", Key=raw_key)
raw_meta_after = (raw_head_after["ContentLength"], raw_head_after["ETag"])
raw_unchanged = raw_meta_before == raw_meta_after

assert raw_row_count == saved_staging_rows + saved_quarantine_rows
assert critical_in_staging == 0
assert mismatched_fin_flag == 0
assert mismatched_mpaa_flag == 0
assert empty_quarantine_reasons == 0
assert raw_unchanged

render_table(
    ["Raw", "Staging", "Quarantine", "Полей staging", "Полей quarantine",
     "Флаг финансов", "Флаг рейтинга", "Крит. нарушений в staging",
     "Raw неизменен"],
    [(
        raw_row_count,
        saved_staging_rows,
        saved_quarantine_rows,
        saved_staging_cols,
        saved_quarantine_cols,
        saved_fin_flagged,
        saved_mpaa_flagged,
        critical_in_staging,
        raw_unchanged,
    )],
)
render_table(
    ["Контрольная проверка", "Расхождений"],
    [
        ("Рассогласование quality_incomplete_financials", mismatched_fin_flag),
        ("Рассогласование quality_nonstandard_rating", mismatched_mpaa_flag),
        ("Пустые причины в saved_quarantine", empty_quarantine_reasons),
    ],
)

Raw,Staging,Quarantine,Полей staging,Полей quarantine,Флаг финансов,Флаг рейтинга,Крит. нарушений в staging,Raw неизменен
7668,7640,28,17,17,2213,419,0,True


Контрольная проверка,Расхождений
Рассогласование quality_incomplete_financials,0
Рассогласование quality_nonstandard_rating,0
Пустые причины в saved_quarantine,0


In [15]:
render_table(
    ["name", "year", "rating", "budget", "gross",
     "quality_incomplete_financials", "quality_nonstandard_rating"],
    conn.execute("""
        SELECT name, year, rating, budget, gross,
               quality_incomplete_financials, quality_nonstandard_rating
        FROM saved_staging
        WHERE quality_incomplete_financials OR quality_nonstandard_rating
        ORDER BY year, name
        LIMIT 2
    """).fetchall(),
)
render_table(
    ["name", "year", "score", "runtime",
     "quality_rule_ids", "quarantine_reason"],
    conn.execute("""
        (SELECT name, year, score, runtime, quality_rule_ids, quarantine_reason
         FROM saved_quarantine WHERE year <= 2019 ORDER BY year, name LIMIT 1)
        UNION ALL
        (SELECT name, year, score, runtime, quality_rule_ids, quarantine_reason
         FROM saved_quarantine WHERE quality_rule_ids LIKE '%;%'
         ORDER BY name LIMIT 1)
    """).fetchall(),
)

name,year,rating,budget,gross,quality_incomplete_financials,quality_nonstandard_rating
Altered States,1980,R,None,19853892.0,True,False
Babylon,1980,None,None,109749.0,True,True


name,year,score,runtime,quality_rule_ids,quarantine_reason
Saw: The Final Chapter,2010,5.6,None,positive_runtime,Отсутствует или неположителен хронометраж runtime
It's Just Us,2020,None,120.0,year_in_study_period;valid_score_and_votes,"Год выпуска вне полных десятилетий 1980–2019; Некорректная или неполная пара (score, votes)"


### Повторное выполнение с чистого ядра

Для проверки идемпотентности перезапустите ядро Jupyter и выполните все ячейки сверху вниз с теми же параметрами (`RAW_S3_URI`, `STUDY_YEAR_MIN = 1980`, `STUDY_YEAR_MAX = 2019`, `STAGING_S3_PREFIX`).

Ожидаемые контрольные показатели для снимка `ingested_on=2026-10-24`: `raw = 7 668`, `staging = 7 640`, `quarantine = 28`, `quality_incomplete_financials = 2 213`, `quality_nonstandard_rating = 419`, критических нарушений в `staging` — `0`.